# FedGATSage — Kaggle Run Notebook
**Graph-based Federated Learning for IoT Intrusion Detection**  
Paper: [Scientific Reports (2025)](https://doi.org/10.1038/s41598-025-25175-1)

This notebook downloads the datasets, installs dependencies, and runs the
experiment by calling the project scripts directly.

> **Prerequisites**: push the full repo (`src/`, `experiments/`, `preprocess_data.py`,
> `fix-NF-TON-IoT-dataset.py`) to your Kaggle notebook via *Add data → Upload*
> or attach it as a dataset.

## 0 — Setup: clone repository & install dependencies
Source: [https://github.com/ioget/FedGATSage-Visiting-code](https://github.com/ioget/FedGATSage-Visiting-code)

In [ ]:
import os

REPO_URL = 'https://github.com/ioget/FedGATSage-Visiting-code'
REPO_DIR = 'FedGATSage-Visiting-code'

# Clone repo if not already present
if not os.path.exists(REPO_DIR):
    print(f'Cloning {REPO_URL} ...')
    os.system(f'git clone {REPO_URL}')
else:
    print(f'{REPO_DIR} already cloned — pulling latest changes ...')
    os.system(f'git -C {REPO_DIR} pull')

# Move into repo root so all script paths resolve correctly
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [ ]:
# Install all dependencies via the project install script
import subprocess, sys
subprocess.run([sys.executable, 'fix-install.py'], check=False)

## 2 — Download datasets from Google Drive

In [ ]:
import os

os.makedirs('data', exist_ok=True)

NF_ID  = '18GMQ1Ncz51t1z2oW-2JJT7CiQFAW7Kkk'
CIC_ID = '120rOiEyxrMQZAFH_KlLMMbK85MQvJkKi'

if not os.path.exists('data/NF-ToN-IoT.csv'):
    print("Downloading NF-ToN-IoT...")
    !gdown {NF_ID} -O data/NF-ToN-IoT.csv
else:
    print("NF-ToN-IoT.csv already present.")

if not os.path.exists('data/CIC-ToN-IoT.csv'):
    print("Downloading CIC-ToN-IoT...")
    !gdown {CIC_ID} -O data/CIC-ToN-IoT.csv
else:
    print("CIC-ToN-IoT.csv already present.")

!ls -lh data/

## 3 — Fix NF-ToN-IoT column schema

In [ ]:
# Renames NF columns to match CIC schema — always re-run to avoid stale/dummy files
import os
# Remove any partial output from a previous failed run
if os.path.exists('data/NF-ToN-IoT-fixed.csv'):
    os.remove('data/NF-ToN-IoT-fixed.csv')
    print('Removed old NF-ToN-IoT-fixed.csv — regenerating...')
!python fix-NF-TON-IoT-dataset.py


## 4 — Preprocess: split into federated client files
Creates `data/cic/` and `data/nf/` each containing
`client_1.csv` … `client_5.csv` + `test.csv`.

In [ ]:
# CIC-ToN-IoT
!python preprocess_data.py \
    --input_file data/CIC-ToN-IoT.csv \
    --output_dir data/cic \
    --num_clients 5 \
    --seed 42

In [ ]:
# NF-ToN-IoT (uses the fixed file)
!python preprocess_data.py \
    --input_file data/NF-ToN-IoT-fixed.csv \
    --output_dir data/nf \
    --num_clients 5 \
    --seed 42

In [ ]:
# Verify real class distribution in test sets (must NOT show dummy classes)
import pandas as pd, os

for name, path in [('CIC', 'data/cic/temporal_detector/test.csv'),
                    ('NF',  'data/nf/temporal_detector/test.csv')]:
    if os.path.exists(path):
        df = pd.read_csv(path, usecols=['Attack'])
        counts = df['Attack'].value_counts()
        print(f'\n{name} test set — {len(df):,} rows, {counts.nunique()} classes:')
        print(counts.to_string())
    else:
        print(f'{name}: test.csv not found at {path}')


## 5 — Run experiment: CIC-ToN-IoT
Change `--num_rounds` or remove `--demo_mode` for the full run.

In [ ]:
!python experiments/fedgatsage_experiment.py \
    --data_dir data/cic \
    --dataset cic_ton_iot \
    --num_clients 5 \
    --num_rounds 5 \
    --detector_types temporal content behavioral \
    --device cpu \
    --output_dir results/cic \
    --seed 42 \
    --demo_mode

## 6 — Run experiment: NF-ToN-IoT

In [ ]:
%%bash
cd FedGATSage-Visiting-code && python experiments/fedgatsage_experiment.py --data_dir data/nf --input_file data/NF-ToN-IoT-fixed.csv --dataset nf_ton_iot --num_clients 5 --num_rounds 10 --detector_types temporal content behavioral --device cuda --output_dir results/nf --seed 42


## 7 — Show results

In [ ]:
# ── Results summary ─────────────────────────────────────────────────────────
import json, glob, os

result_files = sorted(glob.glob('results/**/*_results.json', recursive=True))
if not result_files:
    print('No result files found yet.')
else:
    for rf in result_files:
        with open(rf) as f:
            data = json.load(f)
        ev = data.get('final_results', {}).get('evaluation', {})
        cfg = data.get('final_results', {}).get('configuration', {})
        tr  = data.get('final_results', {}).get('training', {})

        print(f"\n{'='*65}")
        print(f"  Experiment : {data.get('experiment_name', rf)}")
        print(f"  Total time : {data.get('total_time', 0)/60:.1f} min")
        print(f"  Rounds     : {cfg.get('num_rounds','?')}  |  Clients: {cfg.get('num_clients','?')}  |  Classes: {cfg.get('num_classes','?')}")
        print(f"  Detectors  : {cfg.get('detector_types',[])}")
        print(f"{'='*65}")

        if ev:
            print(f"  Accuracy          : {ev.get('accuracy',   'N/A')}")
            print(f"  Balanced Accuracy : {ev.get('balanced_accuracy', 'N/A')}")
            print(f"  Macro F1          : {ev.get('macro_f1',   'N/A')}")
            print(f"  Weighted F1       : {ev.get('weighted_f1','N/A')}")

            pcd = ev.get('per_class_detailed', {})
            if pcd:
                print(f"\n  {'Class':<18} {'F1':>6}  {'Precision':>9}  {'Recall':>6}  {'Support':>7}")
                print(f"  {'-'*55}")
                for cls, m in sorted(pcd.items()):
                    print(f"  {cls:<18} {m['f1']:>6.4f}  {m['precision']:>9.4f}  {m['recall']:>6.4f}  {str(m.get('support','?')):>7}")
        else:
            print('  No evaluation results found.')

        losses = tr.get('training_losses', [])
        if losses:
            print(f"\n  Training loss  : start={losses[0]:.4f}  end={losses[-1]:.4f}  min={min(losses):.4f}")
        print()


In [ ]:
# ── Display all saved plots ──────────────────────────────────────────────────
from IPython.display import Image, display
import glob

plot_names = [
    'training_progress.png',      # loss + round-time curves
    'per_class_performance.png',  # per-class F1 bar chart
    'confusion_matrix.png',       # confusion matrix
]

found = sorted(glob.glob('results/**/*.png', recursive=True))
if not found:
    print('No PNG files found yet — run the experiment first.')
else:
    for img in found:
        print(f'\n── {img} ──')
        display(Image(img))


In [ ]:
# ── List saved model checkpoints ─────────────────────────────────────────────
import glob, os

model_files = sorted(glob.glob('results/**/models/*.pt', recursive=True))
if not model_files:
    print('No model checkpoints found yet.')
else:
    print(f'Found {len(model_files)} model checkpoint(s):\n')
    total_mb = 0
    for mf in model_files:
        size_mb = os.path.getsize(mf) / 1024**2
        total_mb += size_mb
        print(f'  {mf:<70}  {size_mb:.2f} MB')
    print(f'\n  Total: {total_mb:.2f} MB')
    print('\nTo reload a model:')
    print('  import torch')
    print('  ckpt = torch.load("<path>.pt", map_location="cpu")')
    print('  model.load_state_dict(ckpt["state_dict"])')
